# Comparison with TreeCorr

[TreeCorr](https://rmjarvis.github.io/TreeCorr/) is the standard tool for two-point correlation functions in astronomy. The two-point structure function follows from it as $\mathrm{SF}(r) = 2\langle f^2\rangle - 2\langle f_1 f_2\rangle$, using an `NKCorrelation` and a `KKCorrelation`. This page compares the two codes on the same mock catalogue: whether they agree, how fast they are, and what each offers.

Timings depend on the machine; this page is executed when the documentation is built.

In [ ]:
import time
import warnings
from collections.abc import Callable

import astropy.units as u
import numpy as np
import treecorr
from astropy.coordinates import SkyCoord

from structurefunction import structure_function

warnings.filterwarnings("ignore", module="treecorr")


def mock(
    n: int, seed: int = 1
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """All-sky sources with a smooth RM pattern plus noise."""
    rng = np.random.default_rng(seed)
    lon = rng.uniform(0, 360, n)
    lat = np.degrees(np.arcsin(rng.uniform(-1, 1, n)))
    rm = 20 * np.sin(np.radians(lon) * 5) * np.cos(np.radians(lat) * 3)
    rm_err = np.full(n, 2.0)
    return lon, lat, rm + rng.normal(0, rm_err), rm_err


def treecorr_sf(
    lon: np.ndarray,
    lat: np.ndarray,
    values: np.ndarray,
    edges: np.ndarray,
    bin_slop: float | None = None,
    npatch: int = 1,
) -> np.ndarray:
    """SF = 2 <f^2> - 2 <f1 f2> from TreeCorr, with the same bins."""
    k = values - values.mean()
    config = {
        "min_sep": edges[0],
        "max_sep": edges[-1],
        "nbins": len(edges) - 1,
        "sep_units": "deg",
        "metric": "Arc",
    }
    if bin_slop is not None:
        config["bin_slop"] = bin_slop
    positions = {"ra": lon, "dec": lat, "ra_units": "deg", "dec_units": "deg"}
    if npatch > 1:
        config["var_method"] = "jackknife"
        first = treecorr.Catalog(k=k, npatch=npatch, **positions)
        second = treecorr.Catalog(
            k=k**2, patch_centers=first.patch_centers, **positions
        )
    else:
        first = treecorr.Catalog(k=k, **positions)
        second = treecorr.Catalog(k=k**2, **positions)
    kk = treecorr.KKCorrelation(config)
    kk.process(first)
    nk = treecorr.NKCorrelation(config)
    nk.process(first, second)
    if npatch > 1:
        treecorr.estimate_multi_cov(
            [nk, kk], "jackknife", func=lambda c: 2 * c[0].xi - 2 * c[1].xi
        )
    return 2 * nk.xi - 2 * kk.xi


def timed[T](call: Callable[[], T]) -> tuple[T, float]:
    """Result of ``call`` and the seconds it took."""
    start = time.perf_counter()
    result = call()
    return result, time.perf_counter() - start


# Compile our kernels once so timings exclude it.
lon, lat, rm, rm_err = mock(2000, seed=0)
_ = structure_function(rm, rm_err, SkyCoord(lon * u.deg, lat * u.deg), bins=4)

## Same answer

With exact binning (`bin_tolerance=0` here, `bin_slop=0` in TreeCorr) both codes give the same structure function before the error correction. Our structure function also has the measurement-error term removed, so we compare `sf + noise_bias`. The outermost bins differ only in how pairs exactly on the range limits are counted, so they are left out.

In [ ]:
lon, lat, rm, rm_err = mock(20_000)
coords = SkyCoord(lon * u.deg, lat * u.deg)
edges = np.geomspace(0.5, 120, 11)
ours = structure_function(
    rm,
    rm_err,
    coords,
    bins=edges * u.deg,
    bin_tolerance=0,
    jk_regions=1,
    error_method="none",
)
theirs = treecorr_sf(lon, lat, rm, edges, bin_slop=0)
interior = slice(1, -1)
difference = (theirs - (ours.sf + ours.noise_bias))[interior] / ours.sf[interior]
print(np.array2string(difference, precision=2))
assert np.all(np.abs(difference) < 1e-6)

## Speed with error bars

Both codes estimate the covariance between bins with a jackknife over 64 sky regions (TreeCorr calls them patches). TreeCorr processes every pair of patches separately. We add each pair's contribution to the regions it touches in the same pass as the structure function, so the jackknife costs almost nothing extra.

In [ ]:
lon, lat, rm, rm_err = mock(10_000)
coords = SkyCoord(lon * u.deg, lat * u.deg)
ours, t_ours = timed(
    lambda: structure_function(rm, rm_err, coords, bins=15, jk_regions=64)
)
edges = ours.bin_edges.value
_, t_theirs = timed(lambda: treecorr_sf(lon, lat, rm, edges, npatch=64))
print(f"structurefunction, 64-region jackknife: {t_ours:6.2f} s")
print(f"TreeCorr,          64-patch jackknife:  {t_theirs:6.2f} s")

The gap widens with more sources: TreeCorr's time grows faster than $N^2$ here, because each of the $64 \times 65 / 2$ patch pairs is processed on its own.

## Speed without error bars

Without error bars the comparison is closer. Exact against exact, the two are about as fast. TreeCorr's default approximate binning is several times faster than ours, but less accurate, as the last column shows. Speeding up our approximate mode is planned.

In [ ]:
lon, lat, rm, rm_err = mock(50_000)
coords = SkyCoord(lon * u.deg, lat * u.deg)
exact, t_exact = timed(
    lambda: structure_function(
        rm, rm_err, coords, bins=15, bin_tolerance=0, jk_regions=1, error_method="none"
    )
)
edges = exact.bin_edges.value
approx, t_approx = timed(
    lambda: structure_function(
        rm, rm_err, coords, bins=15, jk_regions=1, error_method="none"
    )
)
tc_exact, t_tc_exact = timed(lambda: treecorr_sf(lon, lat, rm, edges, bin_slop=0))
tc_approx, t_tc_approx = timed(lambda: treecorr_sf(lon, lat, rm, edges))
reference = (exact.sf + exact.noise_bias)[interior]


def worst(values: np.ndarray) -> float:
    """Largest relative difference from the exact result."""
    return float(np.max(np.abs(values[interior] - reference) / np.abs(reference)))


print(f"{'':28s} {'time':>8s}  {'worst relative error':>20s}")
print(f"{'ours, exact':28s} {t_exact:7.2f}s")
approx_error = worst(approx.sf + approx.noise_bias)
print(f"{'ours, bin_tolerance=0.05':28s} {t_approx:7.2f}s  {approx_error:20.1e}")
print(f"{'TreeCorr, bin_slop=0':28s} {t_tc_exact:7.2f}s  {worst(tc_exact):20.1e}")
print(
    f"{'TreeCorr, default bin_slop':28s} {t_tc_approx:7.2f}s  {worst(tc_approx):20.1e}"
)

## Precision when large scales dominate

TreeCorr's form, $2\langle f^2\rangle - 2\langle f_1 f_2\rangle$, subtracts two large numbers. When large-scale structure dominates the variance, the small-scale structure function is a tiny difference between them and loses precision. We sum $(f_1 - f_2)^2$ directly (in groups, from each group's mean and spread), so every term is positive and nothing cancels.

Here a strong gradient across a 20-degree field dominates, and the structure function at the smallest separations is about $10^{-6}$ of the total variance.

In [ ]:
rng = np.random.default_rng(0)
n = 30_000
lon = rng.uniform(0, 20, n)
lat = rng.uniform(-10, 10, n)
coords = SkyCoord(lon * u.deg, lat * u.deg)
values = 1000 * lon + np.sin(np.radians(lon) * 600) * np.cos(np.radians(lat) * 600)
edges = np.geomspace(0.01, 10, 10)
exact = structure_function(
    values,
    None,
    coords,
    bins=edges * u.deg,
    bin_tolerance=0,
    jk_regions=1,
    error_method="none",
)
approx = structure_function(
    values, None, coords, bins=edges * u.deg, jk_regions=1, error_method="none"
)
for label, sf in (
    ("ours, bin_tolerance=0.05", approx.sf),
    ("TreeCorr, bin_slop=0", treecorr_sf(lon, lat, values, edges, bin_slop=0)),
    ("TreeCorr, default", treecorr_sf(lon, lat, values, edges)),
):
    relative = (sf - exact.sf) / exact.sf
    print(f"{label:26s}", " ".join(f"{x:8.1e}" for x in relative))

Even TreeCorr's exact binning is off by about $6 \times 10^{-4}$ in the smallest bin, purely from the cancellation.

## Features

| | structurefunction | TreeCorr |
|---|---|---|
| Two-point structure function | yes | via `NK` and `KK` correlations |
| Three-point (second-difference) structure function | yes | no |
| Measurement-error correction, pair by pair | yes | no |
| Positional errors | yes | no |
| Jackknife / bootstrap covariance | single pass | per patch pair |
| Fits, model comparison (BIC, evidence) | yes | no |
| Sky maps of structure functions | yes | no |
| Astropy `Quantity` and any `SkyCoord` frame | yes | arrays in degrees or radians |
| Other correlations (shear, counts, three-point) | no | yes |
| Fastest approximate two-point counts | not yet | yes |
| Wheels for Windows and Linux ARM | yes | no |

Use TreeCorr for general correlation functions. Use this package for structure functions with error bars, the three-point stencil, error and positional-error corrections, fitting and sky maps.